# Working with OpenLLM in Pixeltable

Pixeltable's OpenLLM integration lets you call the AI subscriptions, provider API keys, and local
models you connect to OpenLLM through one OpenAI-compatible endpoint.

### Prerequisites

- The OpenLLM daemon running on your machine (https://docs.openllm.sh/quickstart)

### Important notes

- Usage counts against the subscriptions and provider plans you connect to OpenLLM.
- Be mindful of sensitive data and consider security measures when integrating with external
  services.

OpenLLM uses the OpenAI SDK as its Python API, so we need to install it in addition to Pixeltable.

In [ ]:
%pip install -qU pixeltable openai

By default, Pixeltable calls the local OpenLLM daemon at `http://127.0.0.1:8787/v1`, which needs no
API key. To use another endpoint, set `OPENLLM_BASE_URL`, and set `OPENLLM_API_KEY` if that endpoint
requires authentication.

Now let's create a Pixeltable directory to hold the tables for our demo.

In [1]:
import pixeltable as pxt

# Remove the 'openllm_demo' directory and its contents, if it exists
pxt.drop_dir('openllm_demo', force=True)
pxt.create_dir('openllm_demo')

Connected to Pixeltable database at: postgresql+psycopg://postgres:***@/pixeltable?host=%2Fhome%2Fuser%2F.pixeltable%2Fpgdata
Created directory 'openllm_demo'.


## Responses

Create a table with an input column, then add a computed column that calls OpenLLM through the
Responses API. The `plus` model is a fallback alias: OpenLLM moves to the next model in the alias's
chain when a provider is rate-limited or unavailable. You can also use any model ID listed by the
OpenLLM `models` endpoint.

In [2]:
from pixeltable.functions import openllm

t = pxt.create_table('openllm_demo/chat', {'input': pxt.String})

msgs = [{'role': 'user', 'content': t.input}]
t.add_computed_column(output=openllm.responses(msgs, model='plus'))

Created table 'chat'.
Added 0 column values with 0 errors in 0.00 s


No rows affected.

In [3]:
# Parse the response text into a new column
t.add_computed_column(response=t.output.output[0].content[0].text)

Added 0 column values with 0 errors in 0.00 s


No rows affected.

In [4]:
t.insert(
    [
        {'input': 'What is the capital of France? Answer in one word.'},
        {'input': 'Explain what a computed column is in one sentence.'},
    ]
)
t.select(t.input, t.response).show()

Inserted 2 rows with 0 errors in 2.89 s (0.69 rows/s)


input,response
What is the capital of France? Answer in one word.,Paris
Explain what a computed column is in one sentence.,A computed column is a database table column whose value is automatically calculated from an expression or other column values rather than stored directly.


## Chat completions

`chat_completions()` takes messages in the OpenAI chat format, for code that already uses it.

In [5]:
t.add_computed_column(
    chat_output=openllm.chat_completions(messages=msgs, model='plus')
)
t.add_computed_column(
    chat_response=t.chat_output.choices[0].message.content
)
t.select(t.input, t.chat_response).show()

Added 2 column values with 0 errors in 2.42 s (0.83 rows/s)
Added 2 column values with 0 errors in 0.01 s (276.66 rows/s)


input,chat_response
What is the capital of France? Answer in one word.,Paris
Explain what a computed column is in one sentence.,A computed column is a database table column whose value is automatically calculated from an expression or other column values rather than stored directly.


### Learn more

To learn more about advanced techniques like RAG operations in Pixeltable, check out the
[RAG Operations in Pixeltable](https://docs.pixeltable.com/howto/use-cases/rag-operations)
tutorial.